In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit SDK プリミティブによる厳密シミュレーション

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以上を使用することを推奨します。

```
qiskit[all]~=2.3.0
```
</details>
Qiskit SDK のリファレンスプリミティブは、ローカルの状態ベクトルシミュレーションを実行します。これらのシミュレーションはデバイスノイズのモデリングをサポートしていませんが、より高度なシミュレーション技術（[Qiskit Aer の使用](./simulate-stabilizer-circuits)）や実機での実行（[Qiskit Runtime プリミティブ](primitives)）を検討する前に、アルゴリズムを素早くプロトタイピングするのに役立ちます。

Estimator プリミティブは回路の期待値を計算でき、Sampler プリミティブは回路の出力分布からサンプリングできます。

以下のセクションでは、リファレンスプリミティブを使用してワークフローをローカルで実行する方法を説明します。

## リファレンス Estimator の使用
ローカルの状態ベクトルシミュレーター上で動作する `qiskit.primitives` の `EstimatorV2` リファレンス実装は、[`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator) クラスです。回路、オブザーバブル、パラメーターを入力として受け取り、ローカルで計算された期待値を返します。

以下のコードは、続く例で使用する入力を準備します。オブザーバブルの期待される入力型は [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp) です。この例の回路はパラメーター化されていますが、パラメーター化されていない回路でも Estimator を実行できます。

> **Note:** Estimator に渡す回路には、**測定**を含めては**なりません**。

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** Qiskit Runtime プリミティブのワークフローでは、回路とオブザーバブルを QPU がサポートする命令のみを使用するように変換する必要があります（*命令セットアーキテクチャ（ISA）* 回路およびオブザーバブルと呼ばれます）。リファレンスプリミティブはローカルの状態ベクトルシミュレーションに依存するため、抽象的な命令をそのまま受け取りますが、回路の最適化という観点から、回路をトランスパイルすることは依然として有益な場合があります。
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### Estimator の初期化
[`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator) をインスタンス化します。

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### 実行と結果の取得
この例では、1 つの回路（[`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) 型）と 1 つのオブザーバブルのみを使用します。

[`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run) メソッドを呼び出して推定を実行します。このメソッドは [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob) オブジェクトのインスタンスを返します。[`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result) メソッドを使用して、ジョブから結果（[`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult) オブジェクトとして）を取得できます。

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### 結果から期待値を取得する
プリミティブの結果は [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) オブジェクトの配列を出力します。配列の各要素は `PubResult` オブジェクトであり、その data には PUB 内のすべての回路-オブザーバブルの組み合わせに対応する評価値の配列が含まれています。

最初の（この場合、唯一の）回路評価の期待値とメタデータを取得するには、PUB 0 の評価 [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) にアクセスする必要があります。

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### Estimator の実行オプションの設定
デフォルトでは、リファレンス Estimator は [`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector) クラスに基づく厳密な状態ベクトル計算を実行します。
ただし、これを変更してサンプリングオーバーヘッド（「ショットノイズ」とも呼ばれます）の影響を導入することができます。

Estimator は `precision` 引数を受け取ります。この引数は、プリミティブ実装が期待値の推定に対して目標とするエラーバーを表します。これはサンプリングオーバーヘッドであり、`.run()` メソッド内でのみ定義されます。これにより、PUB レベルまで細かくオプションを調整できます。

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

完全な例については、[プリミティブの例](primitives-examples#estimator-examples) ページをご覧ください。

## リファレンス Sampler の使用
`qiskit.primitives` の `SamplerV2` リファレンス実装は [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) クラスです。回路とパラメーターを入力として受け取り、出力確率分布からサンプリングした結果を出力状態の準確率分布として返します。

以下のコードは、続く例で使用する入力を準備します。これらの例では単一のパラメーター化された回路を実行しますが、パラメーター化されていない回路でも Sampler を実行できます。

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** Sampler に渡す量子回路には、必ず測定を含める**必要があります**。

> **Tip:** Qiskit Runtime プリミティブのワークフローでは、回路を QPU がサポートする命令のみを使用するように変換する必要があります（ISA 回路と呼ばれます）。リファレンスプリミティブはローカルの状態ベクトルシミュレーションに依存するため、抽象的な命令をそのまま受け取りますが、回路の最適化という観点から、回路をトランスパイルすることは依然として有益な場合があります。
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### `SamplerV2` の初期化
[`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) をインスタンス化します。

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### 実行と結果の取得

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


プリミティブは複数の PUB を入力として受け取り、各 PUB はそれぞれの結果を持ちます。そのため、さまざまなパラメーター/オブザーバブルの組み合わせで異なる回路を実行し、PUB の結果を取得できます。

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>